# Holdout Backtesting


## Load Backtest Inputs

- **Purpose:** Load and validate inputs for comparing three tuned meta families with one selected primary configuration; run meta modeling first. This notebook is independent of other backtesting notebooks.
- **Input:** Prepared events, split manifest, primary model, meta-model bundle, saved predictions, and market data.
- **Output:** Validated backtest inputs in memory for the following prediction, sizing, simulation, and statistics stages.
- **STRATEGY:** `market_synchronous` — Choose the matching model artifacts and execution rules.
- **PARTITION:** `holdout` — Use the fixed holdout evaluation window.
- **PERIOD:** `2025-02-01_2025-12-31` — Identify the shared research artifacts; raw preparation data may start earlier.
- **MANIFEST_PATH:** `data/preprocessing/universe/sp500_2025.csv` — Fix the research universe across stages.
- **EXPECTED_SECURITIES:** `503` — Validate the manifest size before processing.
- **DATA_START:** `2025-01-01 UTC` — Include January preparation observations before the research period.
- **END:** `2026-01-01 UTC` — Set the exclusive data-request upper bound.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from src.backtesting.backtesting_workflow import (
    load_backtest_inputs, development_predictions, holdout_predictions,
    prepare_bet_sizing, simulate_portfolios, summarize_backtests, save_backtest_results,
)
from src.backtesting.portfolio_management import PortfolioSettings
from src.preprocessing.market_data import ResearchPaths

PROJECT_ROOT = Path.cwd().resolve().parents[2]
STRATEGY = "market_synchronous"
PARTITION = "holdout"
PERIOD = "2025-02-01_2025-12-31"
MANIFEST_PATH = PROJECT_ROOT / "data/preprocessing/universe/sp500_2025.csv"
EXPECTED_SECURITIES = 503
DATA_START = pd.Timestamp("2025-01-01", tz="UTC")
END = pd.Timestamp("2026-01-01", tz="UTC")
HORIZON_BARS = 1000
settings = PortfolioSettings(
    k=5, step_size=0.10, initial_aum=100_000.0,
    broker_fee_bps=1.0, slippage_bps=1.0, half_life_minutes=390.0,
)
evaluation_settings = dict(
    annual_risk_free_rate=0.03, periods_per_year=365.25,
    annualized_benchmark_sharpe_ratio=1.0,
)
paths = ResearchPaths(PROJECT_ROOT, model_kind="market", period=PERIOD)

In [ ]:
inputs = load_backtest_inputs(
    paths, settings, strategy=STRATEGY, partition=PARTITION,
    manifest_path=MANIFEST_PATH, expected_securities=EXPECTED_SECURITIES,
    data_start=DATA_START, end=END, horizon_bars=HORIZON_BARS,
)


## Holdout Predictions

- **Purpose:** Validate saved holdout predictions without refitting, tuning, or selecting a winner on holdout.
- **Input:** Family-tagged holdout predictions, fixed event/partition keys, and canonical primary predictions.
- **Output:** Validated predictions in memory and displayed coverage summaries; missing/duplicate events, mixed partitions, or inconsistent primary predictions are rejected.


In [ ]:
validation_settings = None
splits = None
predictions = holdout_predictions(inputs)
display(pd.DataFrame([
    {"model_family": family, "events": len(frame),
     "start": frame.index.get_level_values("event_start").min(),
     "end": frame.index.get_level_values("event_start").max(),
     "mean_meta_probability": frame.meta_probability.mean(), "take_rate": frame.meta_action.mean()}
    for (family, path), frame in predictions.items()
]))


## Bet Sizing

- **Purpose:** Select the latest eligible signal per symbol and rank take probabilities at scheduled open/close entries, with a one-minute information cutoff.
- **Input:** Family/path predictions and the session entry/exit schedule.
- **Output:** In-memory selections with equal weights within each side (half the book per side); exclude entries without an exit inside evaluation bounds.
- **k:** `5` — Select at most five symbols per side in each entry round.


In [ ]:
prepared = prepare_bet_sizing(inputs, predictions, splits)
for (family, path), item in prepared.items():
    print(f"{family}: {path}")
    if item["selected"] is not None:
        display(item["selected"][["symbol", "group", "entry", "exit", "target_weight"]].head())
    else:
        display(item["events"][["symbol", "event_start", "event_signal"]].head())
        display(item["calibration"].head())


## Portfolio Management

- **Purpose:** Execute one account per family/path once, funding open/close books independently; open entries exit that close, close entries exit the next close.
- **Input:** Prepared selections and session-boundary prices.
- **Output:** In-memory ledgers, trades, exposures, and exclusions, with displayed account summaries.
- **initial_aum:** `100_000.0` — Set the common starting account capital.
- **broker_fee_bps:** `1.0` — Charge the assumed broker fee on traded notional.
- **slippage_bps:** `1.0` — Charge the assumed execution slippage on traded notional.
- **HORIZON_BARS / step_size / half_life_minutes:** `1000 / 0.10 / 390.0` — Retained shared settings; synchronous execution uses scheduled exits and equal weights, not these controls.


In [ ]:
accounts = simulate_portfolios(inputs, prepared)
display(pd.DataFrame([
    {"model_family": family, "path": path, "observations": len(result["ledger"]),
     "final_aum": result["ledger"].aum.iloc[-1], "excluded_events": len(result["exclusions"])}
    for (family, path), result in accounts.items()
]))


## Backtest Statistics

- **Purpose:** Summarize account, classification, and holding metrics without resimulating.
- **Input:** Existing accounts, out-of-sample predictions, and SPY benchmark returns.
- **Output:** Displayed statistics and account curves; save `statistics.parquet`, `account.parquet`, and `settings.json` under `data/backtest_results/market_synchronous/holdout`. Detailed trades and intermediate predictions remain in memory.
- **annual_risk_free_rate:** `0.03` — Use the assumed annual risk-free rate in risk-adjusted statistics.
- **periods_per_year:** `365.25` — Annualize the calendar-day account series.
- **annualized_benchmark_sharpe_ratio:** `1.0` — Set the reference Sharpe for probabilistic Sharpe evaluation.


In [ ]:
statistics = summarize_backtests(inputs, predictions, accounts, **evaluation_settings)

In [ ]:
display_values = statistics["value"].astype(object).where(
    statistics["unit"].ne("timestamp"), statistics["timestamp"],
)
display(statistics.assign(display_value=display_values).pivot(
    index=["section", "entity", "metric"], columns=["model_family", "path"],
    values="display_value",
))
if PARTITION == "development":
    display(statistics.groupby(["model_family", "section", "entity", "metric"])["value"].describe())

fig, ax = plt.subplots(figsize=(12, 5))
for (family, path), result in accounts.items():
    ledger = result["ledger"]
    ax.plot(ledger.timestamp, ledger.aum / settings.initial_aum - 1, label=f"{family}: {path}")
ax.set(title=f"{STRATEGY}: {PARTITION}", ylabel="Cumulative return", xlabel="Time")
ax.legend(fontsize="small", ncol=3)
fig.tight_layout()
plt.show()

In [ ]:
result_directory = save_backtest_results(
    inputs, statistics, accounts,
    evaluation_settings=evaluation_settings, validation_settings=validation_settings,
)
print(result_directory)
